# ML Assignment 2: train the models on Google Colab

This notebook trains the five classifiers in this project (Logistic Regression, Decision Tree, KNN, Gaussian Naive Bayes, SVM) on the Breast Cancer Wisconsin (Diagnostic) dataset. It also checks that the Streamlit app runs, then packages the results for download.

**Run the cells top to bottom** (Runtime → Run all works too, but cell 2 waits for your file upload). A CPU runtime is enough; training takes about a minute.

| Step | What it does |
|---|---|
| 1 | Show library versions |
| 2 | Upload the project (`colab_upload.zip`) |
| 3 | Train, tune and evaluate the five models |
| 4 | Print the results (copy them for the README observations) |
| 5 | Test the Streamlit app headlessly |
| 6 | Pin `requirements.txt` to the versions used for training |
| 7 | Download `trained_artifacts.zip` |

> Colab deletes files when the runtime disconnects, so finish with step 7 before you close the tab.

## 1. Check the environment
Colab already ships scikit-learn, pandas, numpy and joblib, so training needs no installs. Note the **Python version**: choose the same one when you deploy to Streamlit Cloud.

In [ ]:
import platform, sklearn, numpy, pandas, joblib

print(f"Python       {platform.python_version()}")
print(f"scikit-learn {sklearn.__version__}")
print(f"numpy        {numpy.__version__}")
print(f"pandas       {pandas.__version__}")
print(f"joblib       {joblib.__version__}")

## 2. Upload the project
Run the cell, click **Choose files**, and pick `colab_upload.zip` from your project folder. It contains the code and the dataset.

*Already pushed the project to GitHub?* Replace the upload lines with
`!git clone https://github.com/<your-user>/<your-repo>.git /content/ML_ASSIGNMENT_2`
and keep the rest of the cell.

In [ ]:
import os, shutil, sys, zipfile
from pathlib import Path
from google.colab import files

os.chdir("/content")
PROJECT = Path("/content/ML_ASSIGNMENT_2")
shutil.rmtree(PROJECT, ignore_errors=True)

uploaded = files.upload()                       # choose colab_upload.zip
with zipfile.ZipFile(next(iter(uploaded))) as zf:
    zf.extractall(PROJECT)

# Also accept zips with a top-level folder (e.g. made with Finder's "Compress").
if not (PROJECT / "app.py").exists():
    inner = [p.parent for p in PROJECT.glob("*/app.py")]
    assert inner, "app.py not found in the zip - did you upload the right file?"
    PROJECT = inner[0]

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Project folder:", PROJECT)
for path in sorted(PROJECT.rglob("*")):
    if path.is_file() and "__MACOSX" not in path.parts:
        print("   ", path.relative_to(PROJECT))

## 3. Train and evaluate the five models
Runs `model/train_models.py`:
- stratified 80/20 split (455 train / 114 test rows, `random_state=42`)
- a 5-fold stratified cross-validated grid search per model, on the training rows only
- the six metrics (Accuracy, AUC, Precision, Recall, F1, MCC) on the held-out test rows

It writes `model/*.pkl`, `model/metrics.csv`, `model/metadata.json` and `data/test_data.csv`, and fills in the metrics table in `README.md`.

In [ ]:
!python -m model.train_models

## 4. Review the results
Shows the comparison table, then prints everything as Markdown: metrics, tuned hyper-parameters and each model's confusion matrix. **Copy the printed text.** It's what the *Observations* section of the README is written from.

In [ ]:
import json
import joblib
import pandas as pd
from sklearn.metrics import confusion_matrix

from model.common import FEATURES, MODEL_DIR, MODEL_FILES, TARGET, TEST_DATA_PATH, encode_target
from model.train_models import metrics_markdown

meta = json.loads((MODEL_DIR / "metadata.json").read_text())
metrics = pd.read_csv(MODEL_DIR / "metrics.csv")
display(metrics.set_index("Model").style.format("{:.4f}").highlight_max(axis=0, color="#cde2fb"))

test = pd.read_csv(TEST_DATA_PATH)
X_test, y_test = test[FEATURES], encode_target(test[TARGET])

print(f"Train/test rows: {meta['n_train']}/{meta['n_test']}   versions: {meta['versions']}\n")
print("Test-set metrics (positive class = malignant)\n")
print(metrics_markdown(metrics))
print("\nTuned hyper-parameters and test-set confusion matrices\n")
print(f"| Model | Best params | CV {meta['cv_scoring']} | TN | FP | FN | TP |")
print("|---|---|---:|---:|---:|---:|---:|")
for name, fname in MODEL_FILES.items():
    model = joblib.load(MODEL_DIR / fname)
    tn, fp, fn, tp = confusion_matrix(y_test, model.predict(X_test), labels=[0, 1]).ravel()
    info = meta["models"][name]
    cv = info[f"cv_{meta['cv_scoring']}"]
    print(f"| {name} | {info['best_params']} | {cv:.4f} | {tn} | {fp} | {fn} | {tp} |")

## 5. Test the Streamlit app (headless)
Installs Streamlit and runs `tests/smoke_test_app.py`, which uses Streamlit's `AppTest` (no browser or tunnel needed). It checks that:
- the app runs for every model;
- the metrics it shows, on the built-in test set and on uploaded CSVs, match `model/metrics.csv`;
- invalid uploads are rejected;
- no model is ever trained inside the app.

The last line should read **PASSED**.

pip may print *dependency resolver* warnings about other Colab packages. Those are harmless here.

In [ ]:
!pip install -q "streamlit>=1.32"
!python tests/smoke_test_app.py

## 6. Pin `requirements.txt` to the training versions
The app loads the saved models and never retrains them, and pickled scikit-learn models should be loaded with the same scikit-learn version that saved them. So this rewrites `requirements.txt` to install exactly that version on Streamlit Cloud. If the versions differ, the app shows a warning in the sidebar.

In [ ]:
import json
from pathlib import Path

v = json.loads((PROJECT / "model" / "metadata.json").read_text())["versions"]
np_major = int(v["numpy"].split(".")[0])
pd_major, pd_minor = (int(x) for x in v["pandas"].split(".")[:2])

requirements = f"""streamlit>=1.32
scikit-learn=={v['scikit-learn']}
numpy>={np_major}.0,<{np_major + 1}
pandas>={pd_major}.{pd_minor},<{pd_major + 1}
altair>=5.0
joblib>=1.3
"""
(PROJECT / "requirements.txt").write_text(requirements)
print(requirements)
print(f"When deploying on Streamlit Cloud, pick Python {v['python'].rsplit('.', 1)[0]} under 'Advanced settings'.")

## 7. Download the results
Builds `trained_artifacts.zip` with the trained models, metrics, test split, updated README and pinned requirements, and downloads it to your computer.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

outputs = sorted(PROJECT.glob("model/*.pkl")) + [
    PROJECT / p for p in ("model/metrics.csv", "model/metadata.json", "data/test_data.csv",
                          "README.md", "requirements.txt")
]
archive = Path("/content/trained_artifacts.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in outputs:
        zf.write(path, path.relative_to(PROJECT))
        print("added", path.relative_to(PROJECT))
files.download(str(archive))

## Next steps
1. On your computer, unzip the results over the project folder:
   `unzip -o ~/Downloads/trained_artifacts.zip -d ~/Downloads/ML_ASSIGNMENT_2`
2. Write the *Observations* table in `README.md` (section d.ii) from the step 4 output.
3. Push the project to GitHub, deploy it on Streamlit Community Cloud, and add both links to README section c.
4. Optional: **File → Download → Download .ipynb** saves this notebook with its outputs as a record of the training run. Add it to the repo if you like.